## Importing Packages

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import timm
from torchvision import transforms
import tqdm

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## Data Class Creation Pull

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data = ImageFolder(data_dir, transform=transform)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
    
    @property
    def classes(self):
        return self.data.classes
    

In [ ]:
train_image_dataset = ImageDataset(data_dir=r"C:\Users\Shohan\Desktop\DeepFakeImg\MyLearning\dataset\train")
test_image_dataset = ImageDataset(data_dir=r"C:\Users\Shohan\Desktop\DeepFakeImg\MyLearning\dataset\test")
val_image_dataset = ImageDataset(data_dir=r"C:\Users\Shohan\Desktop\DeepFakeImg\MyLearning\dataset\val")

In [ ]:
len(train_image_dataset), len(test_image_dataset), len(val_image_dataset) 

In [ ]:
image, label = test_image_dataset[300]
print(label)
image


In [ ]:
train_data_dir = train_image_dataset.data.root
val_data_dir = val_image_dataset.data.root
test_data_dir = test_image_dataset.data.root

target_to_class_train = {v : k for k, v in ImageFolder(train_data_dir).class_to_idx.items()}
print(target_to_class_train)
target_to_class_val = {v : k for k, v in ImageFolder(val_data_dir).class_to_idx.items()}
print(target_to_class_val)
target_to_class_test = {v : k for k, v in ImageFolder(test_data_dir).class_to_idx.items()}
print(target_to_class_test)

In [ ]:
image, label = val_image_dataset[100]
print("Image shape:", image.size)
print("Label:", label)
print("Dataset length:", len(val_image_dataset))

In [ ]:
import torchvision.transforms as transforms
import torch # Needed for torch.rand() for dynamic sharpness

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    # 1. Resizing (usually done first)
    transforms.Resize((256, 256)),

    # 2. Geometric Transformations (operate on PIL Images)
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomApply([transforms.ElasticTransform(alpha=250.0, sigma=15.0)], p=0.2),
    # Choose one RandomPerspective; I'll use the p=0.5 as it was first and not commented out
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),

    # 3. Photometric / Color Transformations (operate on PIL Images)
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0)), # Consider adjusting params as discussed earlier

    # Correct way to apply random sharpness/blur
    transforms.RandomApply([
        transforms.RandomAdjustSharpness(sharpness_factor=torch.rand(1).item() * 1.5 + 0.25)
        # This will randomly pick a sharpness factor between 0.25 (blurrier) and 1.75 (sharper)
    ], p=0.2), # Apply this random sharpness/blur with a 20% probability


    # 4. Conversion to Tensor (PIL Image -> Tensor)
    transforms.ToTensor(),

    # 5. Normalization (operate on Tensor)
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),

    # 6. Tensor-based Augmentations (operate on Tensor, should come after ToTensor and Normalize)
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
])

# Don't forget your validation/test transform (no random augmentations)
val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


train_dataset = ImageDataset(train_data_dir, train_transform)
val_dataset = ImageDataset(val_data_dir, val_test_transform)
test_dataset = ImageDataset(test_data_dir, val_test_transform)

In [ ]:
train_dataset[300]

## Itarate over dataset

In [ ]:
for image, label in train_dataset:
    break

In [ ]:
train_dataset

## Dataloader

In [ ]:
# train_detaset is the tensor of out train
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
image, label = train_dataset[100]
print("Image size:", image.size)

In [ ]:
image, label = val_dataset[100]
print("Image shape:", image.shape)  # Use .shape for tensor

In [ ]:
for images, labels in train_dataset:
    break

In [ ]:
labels

## Models

In [ ]:
# Basic efficientnet_b0 Model Class.
class DeepFakeClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.4):
        super(DeepFakeClassifier, self).__init__()
        # Where we define all the parts of the model
        self.base_model = timm.create_model('efficientnet_b0', pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1]) 

        enet_out_size = 1280
        # Make a classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),  # Dropout layer for regularization
            nn.Linear(enet_out_size, num_classes)
        )
    
    def forward(self, x):
        # Connect these parts and return the output
        x = self.features(x)
        output = self.classifier(x)
        return output

In [25]:
model = DeepFakeClassifier(num_classes=2)
print(model)

DeepFakeClassifier(
  (base_model): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (co

In [ ]:


#example_out = model(images)
#example_out.shape # [batch_size, num_classes]

example_val_out = model(train_dataset[300][0].unsqueeze(0))  # Unsqueeze to add batch dimension
example_val_out.shape # [batch_size, num_classes]

In [ ]:

train_loader
test_loader
val_loader

In [ ]:
print("Classes:", train_dataset.classes)
print("Number of classes:", len(train_dataset.classes))

## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
import torch

# Ensure label is a tensor and on the same device as example_val_out
single_label = torch.tensor([label], dtype=torch.long, device=example_val_out.device)
criterion(example_val_out, single_label)
print(example_val_out.shape, single_label.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm # Make sure tqdm is installed: pip install tqdm

# Assuming DeepFakeClassifier, train_loader, val_loader, criterion are already defined
# (including the DeepFakeClassifier with the dropout_rate as discussed)

# Simple training loop
num_epochs = 30 # Keeping 30 epochs as per your last run
train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# --- MODEL AND OPTIMIZER SETUP (assuming your DeepFakeClassifier is defined correctly) ---
# Example:
model = DeepFakeClassifier(num_classes=2, dropout_rate=0.4)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0007)
# -------------------------------------------------------------------------------------

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct_train_predictions = 0
    total_train_samples = 0
    
    for images, labels in tqdm.tqdm(train_loader, desc=f'Training Epoch {epoch+1}/{num_epochs}'):
        # Move inputs and labels to the device
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0) # Accumulate loss weighted by batch size
        
        # Calculate predictions and accumulate correct ones
        preds = outputs.argmax(dim=1)
        correct_train_predictions += (preds == labels).sum().item()
        total_train_samples += labels.size(0)
        
    train_loss = running_loss / total_train_samples
    train_losses.append(train_loss)
    train_acc = correct_train_predictions / total_train_samples # Corrected: average over all train samples
    train_accuracies.append(train_acc)
    
    # Validation phase
    model.eval()
    running_loss = 0.0
    correct_val_predictions = 0
    total_val_samples = 0
    
    with torch.no_grad():
        for images, labels in tqdm.tqdm(val_loader, desc=f'Validation Epoch {epoch+1}/{num_epochs}'):
            # Move inputs and labels to the device
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0) # Accumulate loss weighted by batch size
            
            # Calculate predictions and accumulate correct ones
            preds = outputs.argmax(dim=1)
            correct_val_predictions += (preds == labels).sum().item()
            total_val_samples += labels.size(0)
            
    val_loss = running_loss / total_val_samples
    val_losses.append(val_loss)
    val_acc = correct_val_predictions / total_val_samples # Corrected: average over all val samples
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train loss: {train_loss:.4f}, Validation loss: {val_loss:.4f}, train_accuracy: {train_acc:.4f}, val_accuracy: {val_acc:.4f}")


## Loss Visualization

In [ ]:
plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')
plt.legend()
plt.title("Loss over epochs")
plt.show()

## Accuracy Visualization

In [ ]:
plt.plot(train_accuracies, label='Training accuracy')
plt.plot(val_accuracies, label='Validation loss')
plt.legend()
plt.title("Accuracy over epochs")
plt.show()